In [526]:
import pandas as pd
import numpy as np

import sys
sys.path.append('../')

from sqlalchemy import create_engine, text

from config import RUTA_UNIDAD_ONE_DRIVE
from config import RUTA_LOCAL_ONE_DRIVE
from config import POSTGRES_UTEA

USER_DB = POSTGRES_UTEA['USER']
PASS_DB = POSTGRES_UTEA['PASSWORD']
HOST_DB = POSTGRES_UTEA['HOST']
PORT_DB = POSTGRES_UTEA['PORT']
NAME_DB = POSTGRES_UTEA['DATABASE']

ENGINE = create_engine(f'postgresql+psycopg://{USER_DB}:{PASS_DB}@{HOST_DB}:{PORT_DB}/{NAME_DB}')

In [527]:
#path_xlsx_avance = r'G:/Ingenio Azucarero Guabira S.A/UTEA - SEMANAL - AVANCE COSECHA/2024/AVANCE_SEMANAL/AVANCE DE COSECHA V1.xlsx'
path_xlsx_avance = r'G:\Ingenio Azucarero Guabira S.A\UTEA - SEMANAL - AVANCE COSECHA\2026\AVANCE SEMANAL\AVANCE DE COSECHA_V2.xlsx'
xlsx_resumen = pd.read_excel(path_xlsx_avance, sheet_name='AVANCE_FECHAS')
xlsx_resumen

,COD_COS,FECHA,AREA,ESTIMADO,ENTREGADO
0,1,2026-08-08 00:00:00,29.700690,2116.08,2004.17
1,1,2026-08-14 00:00:00,54.820913,2884.46,1982.15
2,1,2026-08-26 00:00:00,36.761325,2013.00,1278.85
3,1,2026-09-03 00:00:00,15.753562,943.64,789.60
4,1,2026-09-09 00:00:00,33.054996,2664.80,2770.85
...,...,...,...,...,...
5978,(en blanco),2026-09-17 00:00:00,NaN,NaN,72.11
5979,(en blanco),2026-09-25 00:00:00,NaN,NaN,78.61
5980,(en blanco),2026-10-03 00:00:00,NaN,NaN,60.29
5981,(en blanco),2026-10-13 00:00:00,NaN,NaN,275.96


In [530]:
xlsx_resumen['COD_COS'] = pd.to_numeric(xlsx_resumen['COD_COS'], errors='coerce')
# Convertir a número (los no válidos se vuelven NaN)
xlsx_resumen['COD_COS'] = pd.to_numeric(xlsx_resumen['COD_COS'], errors='coerce')
# Eliminar filas con NaN
xlsx_resumen = xlsx_resumen.dropna(subset=['COD_COS'])
# Convertir a int
xlsx_resumen['COD_COS'] = xlsx_resumen['COD_COS'].astype(int)
xlsx_resumen

,COD_COS,FECHA,AREA,ESTIMADO,ENTREGADO
0,1,2026-08-08 00:00:00,29.700690,2116.08,2004.17
1,1,2026-08-14 00:00:00,54.820913,2884.46,1982.15
2,1,2026-08-26 00:00:00,36.761325,2013.00,1278.85
3,1,2026-09-03 00:00:00,15.753562,943.64,789.60
4,1,2026-09-09 00:00:00,33.054996,2664.80,2770.85
...,...,...,...,...,...
5972,674,2026-09-03 00:00:00,3.803514,213.76,NaN
5973,674,2026-09-09 00:00:00,4.852329,319.43,NaN
5974,674,2026-09-17 00:00:00,14.206558,924.65,NaN
5975,674,2026-09-25 00:00:00,4.103038,232.37,NaN


In [531]:
xlsx_resumen['ESTIMADO'].sum()

2528946.91

In [532]:
# 1. Intentar convertir a fecha (los no válidos quedan como NaT)
xlsx_resumen['FECHA'] = pd.to_datetime(xlsx_resumen['FECHA'], errors='coerce')
# Eliminar filas con fechas inválidas
xlsx_resumen = xlsx_resumen.dropna(subset=['FECHA'])

In [533]:
def colapsar_tabla(df_param):
    suma_acumulador = [None] * len(df_param)
    acumulador = 0
    contador = 0
    for i, r in df_param.iterrows():
        if(r['AREA'] == 0):
            acumulador += r['ENTREGADO']
            contador += 1
        else:
            acumulador += r['ENTREGADO']
            suma_acumulador[contador] = acumulador
            acumulador = 0
            contador += 1
            continue
    if suma_acumulador[-1] == None:
        suma_acumulador[-1] = acumulador

    df_param['ENTREGAS'] = suma_acumulador
    
    df_param = df_param.dropna(subset=['ENTREGAS'])
    df_param = df_param.drop(columns=['ENTREGADO'])
    #df_param = df_param[df_param['AREA']!=0]
    return df_param

In [534]:
list_cod_cos = list(set(xlsx_resumen['COD_COS']))

In [535]:
list_df = []

In [536]:
for i in list_cod_cos:
    grupo = xlsx_resumen[xlsx_resumen['COD_COS']==i].fillna(0)
    result = colapsar_tabla(grupo)
    list_df.append(result)

In [537]:
df_combinado = pd.concat(list_df, ignore_index=True)

In [538]:
df_combinado.to_excel(r'G:\Ingenio Azucarero Guabira S.A\UTEA - SEMANAL - AVANCE COSECHA\2026\AVANCE SEMANAL\_TEMP_AVANCE_FECHAS.xlsx', index=False, engine='openpyxl', sheet_name='data')
#df_combinado.to_excel('G:/Ingenio Azucarero Guabira S.A/UTEA - SEMANAL - AVANCE COSECHA/2024/AVANCE_SEMANAL/_TEMP_AVANCE_FECHAS.xlsx', index=False, engine='openpyxl', sheet_name='data')

In [539]:
df_combinado.head(2)

,COD_COS,FECHA,AREA,ESTIMADO,ENTREGAS
0,1,2026-08-08,29.700690,2116.08,2004.17
1,1,2026-08-14,54.820913,2884.46,1982.15


In [540]:
df_combinado['ENTREGAS'].sum()

2405516.5100000002

In [541]:
def cargar_avance_fechas_a_db(df):
    df = df.rename(columns={
            'COD_COS': 'cod_cos',
            'FECHA': 'fecha',
            'AREA': 'area',
            'ESTIMADO': 'estimado',
            'ENTREGAS': 'entregas'
    })
    
    df['cod_cos']     = df['cod_cos'].astype('Int64')
    #df['fecha']   = pd.to_datetime(df["fecha"])
    df['area']  = df['area'].astype('float')
    df['estimado']   = df['estimado'].astype('float')
    df['entregas']   = df['entregas'].astype('float')

    #validad divicion entre 0
    df["tch_estimado"] = np.where(df["area"] > 0,
                              df["estimado"] / df["area"],
                              0)
    #validad divicion entre 0
    df["tch_entregas"] = np.where(df["area"] > 0,
                              df["entregas"] / df["area"],
                              0)
    
    df['diferencia_tn'] = df['entregas'] - df['estimado']
    df["porcen_dif"] = np.where(df["entregas"] > 0,
                              df["diferencia_tn"] / df["entregas"],
                              0)

    with ENGINE.begin() as conn:  # Inicia transacción
        # Vaciar la tabla y reiniciar secuencia
        conn.execute(text(f'TRUNCATE TABLE catastro_iag.data_avance_fechas RESTART IDENTITY'))
        
        # Insertar nuevos datos
        df.to_sql(
            name='data_avance_fechas',
            con=conn,  # conexión cruda dentro de la transacción
            schema='catastro_iag',
            if_exists='append',
            index=False,
            method='multi',
            chunksize=1000
        )
    print(f"✅ Se ha actualziado la tabla de codigos cosecha")

In [542]:
df_combinado

,COD_COS,FECHA,AREA,ESTIMADO,ENTREGAS
0,1,2026-08-08,29.700690,2116.08,2004.17
1,1,2026-08-14,54.820913,2884.46,1982.15
2,1,2026-08-26,36.761325,2013.00,1278.85
3,1,2026-09-03,15.753562,943.64,789.60
4,1,2026-09-09,33.054996,2664.80,2770.85
...,...,...,...,...,...
4222,674,2026-08-29,4.680231,324.93,0.00
4223,674,2026-09-03,3.803514,213.76,0.00
4224,674,2026-09-09,4.852329,319.43,0.00
4225,674,2026-09-17,14.206558,924.65,0.00


In [543]:
df_combinado.loc[(df_combinado['COD_COS'] == 41) & (df_combinado['FECHA'] == '2026-06-20'), 'ESTIMADO'] = -25000
df_combinado.loc[(df_combinado['COD_COS'] == 41) & (df_combinado['FECHA'] == '2026-07-07'), 'ESTIMADO'] = -5000
df_combinado.loc[(df_combinado['COD_COS'] == 41) & (df_combinado['FECHA'] == '2026-07-15'), 'ESTIMADO'] = 30000
df_combinado.loc[(df_combinado['COD_COS'] == 30) & (df_combinado['FECHA'] == '2026-07-30'), 'ESTIMADO'] = -30000
df_combinado.loc[(df_combinado['COD_COS'] == 41) & (df_combinado['FECHA'] == '2026-08-14'), 'ESTIMADO'] = -60000
df_combinado.loc[(df_combinado['COD_COS'] == 30) & (df_combinado['FECHA'] == '2026-08-29'), 'ESTIMADO'] = -18000
df_combinado.loc[(df_combinado['COD_COS'] == 45) & (df_combinado['FECHA'] == '2026-09-17'), 'ESTIMADO'] = -28000
df_combinado.loc[(df_combinado['COD_COS'] == 30) & (df_combinado['FECHA'] == '2026-09-25'), 'ESTIMADO'] = -33000
df_combinado.loc[(df_combinado['COD_COS'] == 666) & (df_combinado['FECHA'] == '2026-09-28'), 'ESTIMADO'] = -26000
df_combinado.loc[(df_combinado['COD_COS'] == 30) & (df_combinado['FECHA'] == '2026-10-03'), 'ESTIMADO'] = -10000

In [544]:
df_combinado[df_combinado['COD_COS'] == 41]

,COD_COS,FECHA,AREA,ESTIMADO,ENTREGAS
331,41,2026-05-29,243.218982,18563.30,16926.05
332,41,2026-06-05,307.814831,23641.00,19404.31
333,41,2026-06-20,113.809833,-25000.00,5209.34
334,41,2026-06-30,25.353719,1838.58,2877.09
335,41,2026-07-07,135.007913,-5000.00,13756.86
336,41,2026-07-15,11.533672,30000.00,1657.47
337,41,2026-08-08,129.806245,9826.94,21564.62
338,41,2026-08-14,45.744757,-60000.00,904.82
339,41,2026-10-03,0.000000,0.00,43413.30


In [545]:
df_combinado[df_combinado['COD_COS'] == 30]

,COD_COS,FECHA,AREA,ESTIMADO,ENTREGAS
253,30,2026-05-29,112.378308,9087.45,11175.78
254,30,2026-06-05,103.696746,8196.55,7206.38
255,30,2026-06-20,24.759220,1802.99,1588.18
256,30,2026-06-30,54.657204,4719.13,5408.70
257,30,2026-07-07,105.501008,9181.54,6651.50
258,30,2026-07-15,27.347041,2053.93,7435.53
259,30,2026-07-22,90.815191,7912.90,3846.44
260,30,2026-07-30,50.662044,-30000.00,6384.11
261,30,2026-08-08,71.787848,5141.31,5123.58
262,30,2026-08-14,65.423072,5219.27,5668.17


In [546]:
cargar_avance_fechas_a_db(df_combinado)

✅ Se ha actualziado la tabla de codigos cosecha
